<a href="https://colab.research.google.com/github/SiddartthVS/cloudRenderer/blob/main/blender_render.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup
**Make sure to read the instructions carefully!**

If you have other resources used in the Blender project and chose to *make all paths relative*, pack all of them into a zip archive. Alternatively, you can *pack all external file*.

* `blender_version` : Version of blender used to render the scene. You may define your own blender version.
* `blend_file_path` : Path to the blend file after unpacking the zip archive. If blend file is used, this is automatically ignored.
___
* `upload_type` : Select the type of upload method. `gdrive_relative` pulls everything from the folder specified.
* `drive_path` : Path to your blend/zip file relative to the root of your Google Drive if `google_drive` is selected. Must  state the file and its extension (.zip/.blend) **unless** `gdrive_relative` is selected.
* `url_blend` : Specify the URL to the blend/zip file if `url` is selected.
___
* `animation` : Specify whether animation or still image is rendered. If **still image** is used, put the frame number in `start_frame`.
* `start_frame, end_frame` : Specify the start and end frame for animation. You may put same value such as zero for both input to set the default frame in the blend file.
___
* `download_type` : Select the type of download method. `gdrive_direct` enables the frames to be outputted directly to Google Drive (zipping will be disabled).
* `output_name` : Name of the output frames, **do NOT include .blend!** (## for frame number)
* `zip_files` : Archive multiple animation frames automatically into a zip file.
* `drive_output_path` : Path to your frames/zip file in Google Drive.
___
* `gpu_enabled, cpu_enabled` : Toggle GPU and CPU for rendering. CPU might give a slight boost in rendering time but may varies depend on the project.
* `optix_enable` : Enable OptiX which may boost performance, may be incompatible depending on the version of blender, project and GPU allocated

After you are done, go to Runtime > Run All (Ctrl + F9) and upload your files or have Google Drive authorised below. See the [GitHub repo](https://github.com/syn73/blender-colab) for more information.

In [12]:
blender_version = '5.1.1' #@param ['2.79b', '2.83.20', '2.93.18', '3.3.21', '3.6.23', '4.2.20', '4.5.9', '5.1.1'] {allow-input: true}
blend_file_path = 'path/to/file.blend' #@param {type: 'string'}
#@markdown ---
upload_type = 'google_drive' #@param ['direct', 'google_drive', 'url', 'gdrive_relative'] {allow-input: false}
drive_path = 'Opening1.blend' #@param {type: 'string'}
url_blend = 'https://drive.google.com/file/d/1T1U2XK9AhZkLQMssOb7VQEXgA9Xvscc2/view?usp=drive_link' #@param {type: 'string'}
#@markdown ---
animation = True #@param {type: 'boolean'}
start_frame =  230#@param {type: 'integer'}
end_frame =  235#@param {type: 'integer'}
#@markdown ---
download_type = 'google_drive' #@param ['direct', 'google_drive', 'gdrive_direct'] {allow-input: false}
output_name = 'opening-##' #@param {type: 'string'}
zip_files = False #@param {type: 'boolean'}
drive_output_path = 'Spiderman' #@param {type: 'string'}
#@markdown ---
gpu_enabled = True #@param {type:"boolean"}
optix_enabled = False #@param {type:"boolean"}
cpu_enabled = False #@param {type:"boolean"}

In [13]:
%cd /content

gpu = !nvidia-smi --query-gpu=gpu_name --format=csv,noheader
print("Current GPU: " + gpu[0])

if gpu[0] == "Tesla K80" and optix_enabled:
  print("OptiX disabled because of unsupported GPU")
  optix_enabled = False

/content
Current GPU: Tesla T4


In [14]:
import os

os.environ["LD_PRELOAD"] = ""

!apt remove libtcmalloc-minimal4
!apt install libtcmalloc-minimal4

os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4.5.9"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages will be REMOVED:
  libtcmalloc-minimal4
0 upgraded, 0 newly installed, 1 to remove and 51 not upgraded.
After this operation, 382 kB disk space will be freed.
(Reading database ... 122394 files and directories currently installed.)
Removing libtcmalloc-minimal4:amd64 (2.9.1-0ubuntu3) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_

In [15]:
import shutil
from google.colab import files, drive
uploaded_filename = ""

if upload_type == 'google_drive' or upload_type == 'gdrive_relative' or download_type == 'google_drive' or download_type == 'gdrive_direct':
    drive.mount('/drive')

if upload_type == 'direct':
    uploaded = files.upload()
    for fn in uploaded.keys():
        uploaded_filename = fn
elif upload_type == 'url':
    !wget -nc $url_blend
    uploaded_filename = os.path.basename(url_blend)
elif upload_type == 'google_drive':
    shutil.copy('/drive/MyDrive/' + drive_path, '.')
    uploaded_filename = os.path.basename(drive_path)

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [16]:
!rm -r render
!mkdir render

if upload_type == 'gdrive_relative':
    if not drive_path.endswith('/'):
        drive_path += '/'
    !cp -r '/drive/MyDrive/{drive_path}.' 'render/'
elif uploaded_filename.lower().endswith('.zip'):
    !unzip -o $uploaded_filename -d 'render/'
elif uploaded_filename.lower().endswith('.blend'):
    shutil.copy(uploaded_filename, 'render/')
    blend_file_path = uploaded_filename
else:
    raise SystemExit("Invalid file extension, only .blend and .zip can be uploaded.")

In [17]:
import requests
blender_url_dict = {
    '2.79b'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.79/blender-2.79b-linux-glibc219-x86_64.tar.bz2",
    '2.83.20' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.83/blender-2.83.20-linux-x64.tar.xz" # example
    # The link will be inferred automatically even though the version is not defined here, you may override this behaviour by defining a new version here
    # Add any custom Linux binaries, refer to https://ftp.nluug.nl/pub/graphics/blender/release or other sites that provide direct link download
}

if blender_version in blender_url_dict:
    blender_url = blender_url_dict[blender_version]
else:
    major_minor = ".".join(blender_version.split('.')[:2])
    blender_url = f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender{major_minor}/blender-{blender_version}-linux-x64.tar.xz"

try:
    response = requests.head(blender_url, allow_redirects=True, timeout=10)
    if response.status_code != 200:
        print(f"Download failed for version '{blender_version}'.")
        print("Error downloading: You may need to define the download archive manually above.")
    else:
        base_url = os.path.basename(blender_url)
        print(f"Download URL: {blender_url}")
        print(f"Base filename: {base_url}")
except Exception as e:
    print(f"Error checking URL: {e}")
    print("Error downloading: You may need to define the download archive manually above.")

!mkdir $blender_version
!wget -nc $blender_url
!tar -xkf $base_url -C ./$blender_version --strip-components=1

Streaming output truncated to the last 5000 lines.
tar: lib/usd/usdSkelImaging/resources/shaders/skinning.glslfx: Cannot open: File exists
tar: lib/usd/usdUI/resources/usdUI/schema.usda: Cannot open: File exists
tar: lib/usd/usdUI/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/usdUI/resources/generatedSchema.usda: Cannot open: File exists
tar: lib/usd/usdRiPxrImaging/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/hio/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/usdPhysicsValidators/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/usdSemantics/resources/usdSemantics/schema.usda: Cannot open: File exists
tar: lib/usd/usdSemantics/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/usdSemantics/resources/generatedSchema.usda: Cannot open: File exists
tar: lib/usd/hdStorm/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/usdUtilsValidators/resources/plugInfo.json: Cannot open: File exists
tar: lib/usd/p

In [18]:
# Enable GPU rendering (or add custom properties here)
data = "import re\n"+\
    "import bpy\n"+\
    "scene = bpy.context.scene\n"+\
    "scene.cycles.device = 'GPU'\n"+\
    "prefs = bpy.context.preferences\n"+\
    "prefs.addons['cycles'].preferences.get_devices()\n"+\
    "cprefs = prefs.addons['cycles'].preferences\n"+\
    "print(cprefs)\n"+\
    "for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):\n"+\
    "    try:\n"+\
    "        cprefs.compute_device_type = compute_device_type\n"+\
    "        print('Device found:',compute_device_type)\n"+\
    "        break\n"+\
    "    except TypeError:\n"+\
    "        pass\n"+\
    "for device in cprefs.devices:\n"+\
    "    if not re.match('intel', device.name, re.I):\n"+\
    "        print('Activating',device)\n"+\
    "        device.use = "+str(gpu_enabled)+"\n"+\
    "    else:\n"+\
    "        device.use = "+str(cpu_enabled)+"\n"
with open('setgpu.py', 'w') as f:
    f.write(data)

renderer = "CUDA"
if optix_enabled:
    print("Note: You're currently using OptiX renderer. If an error occurred, the current GPU (e.g. Tesla K80) is not supported and you need to switch back to CUDA.")
    renderer = "OPTIX"

In [21]:
import os, gc
gc.collect()

%cd /content
# We don't want to wipe the output folder if we are resuming/recovering
if not os.path.exists('output'):
    os.makedirs('output')

if not drive_output_path.endswith('/'):
    drive_output_path += '/'

# Final Drive Destination
gdrive_dest = '/drive/MyDrive/' + drive_output_path

if download_type != 'gdrive_direct':
    output_path = '/content/output/' + output_name
else:
    output_path = gdrive_dest + output_name

# Create a more robust helper script
with open('/content/render_layers.py', 'w') as f:
    f.write(f"""
import bpy
import os
import subprocess

scene = bpy.context.scene
base_output = '{output_path}'
gdrive_dest = '{gdrive_dest}'

# Force the user-defined frame range
start = {start_frame}
end = {end_frame}

view_layers = [l.name for l in scene.view_layers if l.use]

for frame in range(start, end + 1):
    scene.frame_set(frame)
    for layer_name in view_layers:
        # Isolate layer
        for l in scene.view_layers:
            l.use = (l.name == layer_name)

        # Define filepath for this specific frame and layer
        # Blender replaces # with frame number
        file_suffix = f"_{{layer_name}}"
        scene.render.filepath = f"{{base_output}}{{file_suffix}}"

        print(f'\\n>>> RENDERING: Frame {{frame}} | Layer: {{layer_name}}')
        bpy.ops.render.render(write_still=True)

        # IMMEDIATE BACKUP TO DRIVE
        # Logic: find the file just saved in /content/output and copy to Drive
        rendered_file = f"/content/output/{{os.path.basename(base_output).replace('##', str(frame).zfill(2))}}{{file_suffix}}.png"
        if os.path.exists(rendered_file):
            subprocess.run(['cp', rendered_file, gdrive_dest])
            print(f'>>> BACKED UP TO DRIVE: {{rendered_file}}')
""")

%cd /content/$blender_version

# Run blender in background mode using our custom frame-looping script
!./blender -b '/content/render/{blend_file_path}' -P "/content/setgpu.py" -P "/content/render_layers.py" -E CYCLES -noaudio -- --cycles-device "{renderer}"

/content
/content/5.1.1
Blender 5.1.1 (hash b70da489d7f4 built 2026-04-14 01:31:31)
00:00.399  blend            | Read blend: "/content/render/Opening1.blend"
00:02.693  cycles           | WARNING HIPEW initialization failed: Error opening HIP dynamic library
<bpy_struct, CyclesPreferences at 0x9c6064e8>
Device found: CUDA
Activating <bpy_struct, CyclesDeviceSettings("Tesla T4") at 0x18128788>
Activating <bpy_struct, CyclesDeviceSettings("Tesla T4") at 0x181288a8>

>>> RENDERING: Frame 230 | Layer: spidey
01:03.261  render           | Saved: '/content/output/opening-##_spidey.png'
>>> BACKED UP TO DRIVE: /content/output/opening-230_spidey.png

>>> RENDERING: Frame 230 | Layer: city1
01:24.999  cycles           | ERROR Image file  does not exist.
^C


In [20]:
%cd /content

path, dirs, files_folder = next(os.walk("output"))
output_folder_name = output_name.replace('#', '') + 'render'

if download_type == 'gdrive_direct':
    pass
elif len(files_folder) == 1:
    render_img = 'output/' + files_folder[0]
    if download_type == 'direct':
        files.download('output/' + files_folder[0])
    else:
        shutil.copy('/content/' + render_img, '/drive/MyDrive/' + drive_output_path)
elif len(files_folder) > 1:
    if zip_files:
        shutil.make_archive(output_folder_name, 'zip', 'output')
    if download_type == 'direct':
        files.download(output_folder_name + '.zip')
    else:
        shutil.copy('/content/' + output_folder_name + ".zip", '/drive/MyDrive/' + drive_output_path)
elif download_type == 'direct':
    for f in files_folder:
        files.download('output/{}'.format(f))
    # Drive, no zip
    else:
        for f in files_folder:
          shutil.copy("/content/output/" + f, '/drive/MyDrive/' + drive_output_path + f)
else:
    raise SystemExit("No frames are rendered.")

/content


FileNotFoundError: [Errno 2] No such file or directory: '/content/opening-render.zip'

## Disclaimer
Google Colab is targeted to researchers and students to run AI/ML tasks, data analysis and education, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies time by time. [Colab Pro and Colab Pro+](https://colab.research.google.com/signup) are available for those who wanted to have more powerful GPU and longer runtimes for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info. In some cases, it might be faster to use an online Blender renderfarm.

## License
```
MIT License

Copyright (c) 2020-2022 ynshung

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```